# 🔄 Basis Agent Workflows met GitHub-modellen (Python)

## 📋 Workflow Orchestratie Tutorial

Deze notebook introduceert de krachtige **Workflow Builder** mogelijkheden van het Microsoft Agent Framework. Leer hoe je geavanceerde, meerstaps agentworkflows maakt die complexe bedrijfsprocessen kunnen afhandelen en meerdere AI-operaties naadloos kunnen coördineren.

## 🎯 Leerdoelen

### 🏗️ **Workflow Architectuur**
- **Workflow Builder**: Ontwerp en orkestreer complexe meerstaps processen
- **Event-Driven Execution**: Afhandelen van workflow events en toestandovergangen
- **Visual Workflow Design**: Maak en visualiseer workflowstructuren
- **GitHub Models Integration**: Maak gebruik van AI-modellen binnen workflowcontexten

### 🔄 **Proces Orchestratie**
- **Sequential Operations**: Keten van meerdere agenttaken in logische volgorde
- **Conditional Logic**: Implementeer beslispunten en vertakkende workflows
- **Error Handling**: Robuuste foutafhandeling en workflow veerkracht
- **State Management**: Volg en beheer de uitvoeringstoestand van workflows

### 📊 **Enterprise Workflow Patronen**
- **Business Process Automation**: Automatiseer complexe organisatorische workflows
- **Multi-Agent Coordination**: Coördineer meerdere gespecialiseerde agenten
- **Scalable Execution**: Ontwerp workflows voor bedrijfsbrede operaties
- **Monitoring & Observability**: Volg workflow prestaties en uitkomsten

## ⚙️ Vereisten & Setup

### 📦 **Benodigde Afhankelijkheden**

Installeer het Agent Framework met workflow mogelijkheden:

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub Models Configuratie**

**Omgeving Setup (.env bestand):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

### 🏢 **Enterprise Use Cases**

**Voorbeelden van bedrijfsprocessen:**
- **Customer Onboarding**: Meerstaps verificatie- en setupworkflows
- **Content Pipeline**: Geautomatiseerde creatie, beoordeling en publicatie van content
- **Data Processing**: ETL-workflows met AI-gestuurde transformatie
- **Quality Assurance**: Geautomatiseerde test- en validatieprocessen

**Voordelen van workflows:**
- 🎯 **Betrouwbaarheid**: Deterministische uitvoering met foutherstel
- 📈 **Schaalbaarheid**: Verwerk grootschalige procesautomatisering
- 🔍 **Observeerbaarheid**: Volledige audit trails en monitoring
- 🔧 **Onderhoudbaarheid**: Visueel ontwerp en modulaire componenten

## 🎨 Workflow Ontwerp Patronen

### Basis Workflow Structuur
```mermaid
graph TD
    A[Start] --> B[Agent Taak 1]
    B --> C{Beslissingspunt}
    C -->|Succes| D[Agent Taak 2]
    C -->|Fout| E[Foutafhandeling]
    D --> F[Einde]
    E --> F
```

**Belangrijke Componenten:**
- **WorkflowBuilder**: Hoofd orchestratie motor
- **WorkflowEvent**: Eventafhandeling en communicatie
- **WorkflowViz**: Visuele workflow representatie en debugging

Laten we je eerste intelligente workflow bouwen! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load GitHub Models API credentials from .env file
load_dotenv()

In [ ]:
# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = await provider.create_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = await provider.create_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
